In [13]:
from import_images import encontrar_imagens_tiff, carregar_imagem_por_indice
from cellpose import models, utils
from skimage.exposure import equalize_adapthist
from skimage.morphology import white_tophat, closing, disk, label
from skimage.measure import regionprops
from scipy.ndimage import gaussian_filter
import numpy as np
import matplotlib.pyplot as plt
import os


class ProcessadorDeImagens:
    def __init__(self, base_dir, modelo='nuclei', canais=[0, 0]):
        self.image_paths = encontrar_imagens_tiff(base_dir)
        self.modelo = models.Cellpose(model_type=modelo)
        self.canais = canais
        self.output_dir = os.path.join(base_dir, "segmentacoes")
        os.makedirs(self.output_dir, exist_ok=True)

    def preprocessar_imagem(self, imagem):
        imagem = gaussian_filter(imagem, sigma=1)
        imagem = white_tophat(imagem, footprint=disk(15))
        imagem = equalize_adapthist(imagem, clip_limit=0.4)
        imagem = (imagem - np.min(imagem)) / (np.max(imagem) - np.min(imagem))
        imagem = closing(imagem, disk(3))
        return imagem

    def filtrar_objetos(self, mascara, imagem_original, limiar_circularidade=0.75, limiar_area=80, fator_intensidade=1.2):
        nova_mascara = np.zeros_like(mascara)
        props = regionprops(label(mascara), intensity_image=imagem_original)

        media_imagem = np.mean(imagem_original)
        index = 1
        for prop in props:
            if prop.perimeter == 0:
                continue

            circularidade = (4 * np.pi * prop.area) / (prop.perimeter ** 2)
            intensidade_media = prop.mean_intensity
            area = prop.area

            if (
                circularidade >= limiar_circularidade and
                area >= limiar_area and
                intensidade_media > media_imagem * fator_intensidade
            ):
                nova_mascara[label(mascara) == prop.label] = index
                index += 1

        return nova_mascara

    def carregar_e_processar(self, indice):
        imagem_original = carregar_imagem_por_indice(self.image_paths, indice)
        if imagem_original is None:
            print("Não foi possível carregar a imagem.")
            return None, None

        print("Pré-processando imagem...")
        imagem = self.preprocessar_imagem(imagem_original)

        print("Segmentando com Cellpose...")
        masks, flows, styles, diams = self.modelo.eval(
            imagem,
            channels=self.canais,
            flow_threshold=1,
            cellprob_threshold=-0.1,
            min_size=300
        )

        print("Filtrando objetos...")
        masks_filtradas = self.filtrar_objetos(
            mascara=masks,
            imagem_original=imagem_original,
            limiar_circularidade=0.75,
            limiar_area=80,
            fator_intensidade=1.2
        )

        n_objetos = len(np.unique(masks_filtradas)) - 1
        print(f"{n_objetos} objetos circulares mantidos")

        outlines = utils.outlines_list(masks_filtradas)
        nome_arquivo = os.path.basename(self.image_paths[indice])
        nome_base = os.path.splitext(nome_arquivo)[0]
        caminho_saida = os.path.join(self.output_dir, f"{nome_base}_segmentado.png")

        fig, ax = plt.subplots(figsize=(6, 6))
        ax.imshow(imagem_original, cmap='gray')
        for o in outlines:
            ax.plot(o[:, 0], o[:, 1], color='red', linewidth=0.5)

        ax.set_title(f"{nome_base} - {n_objetos} objetos circulares")
        ax.axis('off')
        plt.savefig(caminho_saida, dpi=300, bbox_inches='tight')
        plt.close()

        print(f"Segmentação salva em: {caminho_saida}")
        return masks_filtradas, flows


if __name__ == "__main__":
    base_dir = os.getcwd()
    proc = ProcessadorDeImagens(base_dir, modelo='nuclei')
    masks, flows = proc.carregar_e_processar(0)


Imagem carregada: /home/kayllany.oliveira/remote-repos/CellViability/data/17 Resultados HTS SLEV/SLEV HTS TargetMol/S_C1_R1[10279]/2022-04-11T155737Z[10932]/008012-1-001001001.tif - Dimensão: (1024, 1360)
Pré-processando imagem...
Segmentando com Cellpose...
Filtrando objetos...
67 objetos circulares mantidos
Segmentação salva em: /home/kayllany.oliveira/remote-repos/CellViability/segmentacoes/008012-1-001001001_segmentado.png


In [3]:
!pwd

/home/kayllany.oliveira/remote-repos/CellViability


In [8]:
import os
import numpy as np
from tifffile import imread, imwrite
from stardist.models import StarDist2D
from cellpose import models as cp_models
from skimage import exposure, morphology, filters
from skimage.util import img_as_ubyte
from import_images import encontrar_imagens_tiff, carregar_imagem_por_indice


class ComparadorDeSegmentacao:
    def __init__(self, base_dir, canais_cellpose=[0, 0]):
        self.base_dir = base_dir
        self.image_paths = encontrar_imagens_tiff(base_dir)
        self.modelo_stardist = StarDist2D(None, name='2D_versatile_fluo', basedir='/home/kayllany.oliveira/remote-repos/CellViability')
        self.modelo_cellpose = cp_models.Cellpose(gpu=True, model_type='nuclei')
        self.canais_cellpose = canais_cellpose

        # Criar pastas de saída para resultados
        self.output_dir = os.path.join(base_dir, "resultados")
        self.out_stardist = os.path.join(self.output_dir, "stardist")
        self.out_cellpose = os.path.join(self.output_dir, "cellpose")
        os.makedirs(self.out_stardist, exist_ok=True)
        os.makedirs(self.out_cellpose, exist_ok=True)

    def preprocessar_imagem(self, imagem):
        if imagem.ndim == 3:
            imagem = imagem[:, :, 0]  # Assume RGB e usa o canal R
        imagem = exposure.equalize_adapthist(imagem)  # Realce de contraste
        imagem = filters.rank.mean(img_as_ubyte(imagem), morphology.disk(2))  # Suavização
        return imagem

    def segmentar_stardist(self, imagem):
        labels, _ = self.modelo_stardist.predict_instances(imagem)
        return labels

    def segmentar_cellpose(self, imagem):
        masks, _, _, _ = self.modelo_cellpose.eval(imagem, diameter=None, channels=self.canais_cellpose)
        return masks

    def salvar_resultado(self, mascara, indice, sufixo):
        nome_arquivo = os.path.splitext(os.path.basename(self.image_paths[indice]))[0]
        if sufixo == "stardist":
            caminho = os.path.join(self.out_stardist, f"{nome_arquivo}_{sufixo}.tiff")
        else:
            caminho = os.path.join(self.out_cellpose, f"{nome_arquivo}_{sufixo}.tiff")

        imwrite(caminho, mascara.astype(np.uint16))
        print(f"Salvo: {caminho}")

    def processar_todas_as_imagens(self):
        for i in range(len(self.image_paths)):
            print(f"\n🔍 Processando imagem {i+1}/{len(self.image_paths)}: {self.image_paths[i]}")
            imagem = carregar_imagem_por_indice(self.image_paths, i)
            if imagem is None:
                print("⚠️  Imagem não encontrada ou inválida. Pulando.")
                continue

            imagem_pre = self.preprocessar_imagem(imagem)

            # Segmentação com StarDist
            mascara_sd = self.segmentar_stardist(imagem_pre)
            self.salvar_resultado(mascara_sd, i, sufixo="stardist")

            # Segmentação com Cellpose
            mascara_cp = self.segmentar_cellpose(imagem_pre)
            self.salvar_resultado(mascara_cp, i, sufixo="cellpose")

        print("\n✅ Processamento e salvamento concluídos.")


if __name__ == "__main__":
    base_dir = os.getcwd()  # ou forneça o caminho desejado
    comparador = ComparadorDeSegmentacao(base_dir)
    comparador.processar_todas_as_imagens()


I0000 00:00:1748377265.137759 2100950 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 40053 MB memory:  -> device: 0, name: NVIDIA A40, pci bus id: 0000:81:00.0, compute capability: 8.6


Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.479071, nms_thresh=0.3.

🔍 Processando imagem 1/13061: /home/kayllany.oliveira/remote-repos/CellViability/data/17 Resultados HTS SLEV/SLEV HTS TargetMol/S_C1_R1[10279]/2022-04-11T155737Z[10932]/008012-1-001001001.tif
Imagem carregada: /home/kayllany.oliveira/remote-repos/CellViability/data/17 Resultados HTS SLEV/SLEV HTS TargetMol/S_C1_R1[10279]/2022-04-11T155737Z[10932]/008012-1-001001001.tif - Dimensão: (1024, 1360)


base.py (406): Predicting on non-float input... ( forgot to normalize? )
functional.py (238): The structure of `inputs` doesn't match the expected structure.
Expected: ['input']
Received: inputs=Tensor(shape=(1, 1024, 1360, 1))
I0000 00:00:1748377267.674997 2391211 service.cc:152] XLA service 0x7fd70c00b520 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1748377267.675341 2391211 service.cc:160]   StreamExecutor device (0): NVIDIA A40, Compute Capability 8.6
2025-05-27 17:21:07.821530: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
E0000 00:00:1748377268.075019 2391211 cuda_dnn.cc:522] Loaded runtime CuDNN library: 9.1.0 but source was compiled with: 9.3.0.  CuDNN library needs to have matching major version and equal or higher minor version. If using a binary install, upgrade your CuDNN library.  If building from sources, mak

FailedPreconditionError: Graph execution error:

Detected at node StatefulPartitionedCall defined at (most recent call last):
  File "/home/kayllany.oliveira/.local/share/uv/python/cpython-3.10.16-linux-x86_64-gnu/lib/python3.10/runpy.py", line 196, in _run_module_as_main

  File "/home/kayllany.oliveira/.local/share/uv/python/cpython-3.10.16-linux-x86_64-gnu/lib/python3.10/runpy.py", line 86, in _run_code

  File "/home/kayllany.oliveira/remote-repos/CellViability/.venv/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/home/kayllany.oliveira/remote-repos/CellViability/.venv/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance

  File "/home/kayllany.oliveira/remote-repos/CellViability/.venv/lib/python3.10/site-packages/ipykernel/kernelapp.py", line 739, in start

  File "/home/kayllany.oliveira/remote-repos/CellViability/.venv/lib/python3.10/site-packages/tornado/platform/asyncio.py", line 205, in start

  File 

2.19.0
